In [11]:
from config import EXTERNAL_DATASET_INPUTS, INTERNAL_DATASET_INPUTS, Columns, ALIGNMENT_PATH
from data_processing.processing import TextProcessor
from data_processing.alignment import Aligner
from utils import stack_csvs_from_folder, drop_empty_rows
import pandas as pd


def load_data() -> pd.DataFrame:
    df_internal = stack_csvs_from_folder(INTERNAL_DATASET_INPUTS)
    return drop_empty_rows(df_internal)


In [12]:
import pandas as pd

def align(row, guide_by_id):
    oare_id = row["id"]
    
    if oare_id not in guide_by_id:
        doc_row = row.to_dict()
        doc_row["level"] = "document"
        doc_row["alignment_status"] = "NO_GUIDE"
        return [doc_row]

    sent_rows = guide_by_id[oare_id]
    
    # Pre-calculate word count to handle the "last interval" logic
    words_trans = row["transliteration"].split()
    max_words = len(words_trans)
    
    # Get the (start, end) points for slicing
    intervals = slice_transliteration(row["transliteration"], sent_rows)
    
    final_segments = []

    for i, (_, guide_row) in enumerate(sent_rows.iterrows()):
        start_idx, end_idx = intervals[i]

        # Ensure the end_idx defaults to the maximum if it was the last segment
        if end_idx is None or end_idx > max_words:
            end_idx = max_words

        segment = row.to_dict().copy()
        segment["level"] = "sentence"
        segment["sentence_idx"] = i
        
        # The data you'll pass to the LLM
        segment["original_transliteration"] = row["transliteration"]
        segment["slice_interval"] = (start_idx, end_idx)
        segment["guide_translation"] = guide_row["translation"]
        segment["translation"] = row["translation"]
        
        # Add a visual check for your own debugging
        segment["text_at_interval"] = " ".join(words_trans[start_idx:end_idx]) if start_idx is not None else ""
        
        segment["alignment_status"] = "READY_FOR_LLM"
        final_segments.append(segment)

    return final_segments

def slice_transliteration(source_text, alignment_rows):
    words = source_text.split()
    total_len = len(words)
    intervals = []

    for i in range(len(alignment_rows)):
        curr_row = alignment_rows.iloc[i]
        
        # Start point
        if pd.isna(curr_row["first_word_number"]):
            start = None
        else:
            start = int(curr_row["first_word_number"]) - 1

        # End point logic: Look at the next row's start, or use total length
        if i + 1 < len(alignment_rows):
            next_row = alignment_rows.iloc[i + 1]
            if not pd.isna(next_row["first_word_number"]):
                end = int(next_row["first_word_number"]) - 1
            else:
                end = total_len
        else:
            # This is the last interval
            end = total_len

        intervals.append((start, end))
    
    return intervals

In [13]:
sentences_oare = pd.read_csv(ALIGNMENT_PATH)
sentences_oare = sentences_oare.dropna(subset=['translation'])
train_df = load_data()

guide_by_id = {
    k: g.sort_values("first_word_number").reset_index(drop=True)
    for k, g in sentences_oare.groupby("text_uuid")
}

all_rows = []

for _, row in train_df.iterrows():
    processed_rows = align(row, guide_by_id)
    all_rows.extend(processed_rows)

final_df = pd.DataFrame(all_rows)

No empty rows found.
Rows before: 1561, Rows after: 1561


In [17]:
final_df.head()


# To see the distribution of all levels (document vs sentence)
print(final_df['level'].value_counts())

level
document    1309
sentence    1207
Name: count, dtype: int64


In [15]:
def generate_alignment_prompt(oare_id, final_df, train_df):
    # Get the high-quality full translation from the original training data
    full_train_row = train_df[train_df['id'] == oare_id].iloc[0]
    full_train_text = full_train_row['translation']
    orig_trans = full_train_row['transliteration']
    
    # Get the segmented guide rows
    segments = final_df[final_df['id'] == oare_id].sort_values('sentence_idx')
    
    segment_block = ""
    for _, row in segments.iterrows():
        segment_block += (
            f"Segment {row['sentence_idx']}:\n"
            f"- Suggested Interval: {row['slice_interval']}\n"
            f"- Guide Translation: {row['guide_translation']}\n\n"
        )

    prompt = f"""
### ALIGNMENT TASK: {oare_id}

**Full Original Transliteration:**
{orig_trans}

**Full High-Quality Training Translation:**
{full_train_text}

**Proposed Segments to Align:**
{segment_block}

**Instructions:**
Your instructions are the following:
- Align the transliteration to the high-quality translation using the guide translations as a reference for sentence breaks.
- The interval refers to the places where the sentence should break in the original transliteration.
- The intervals might have errors and be slightly misplaced.
- You must return your output in a csv the following format: oare_id,aligned_transliteration,aligned_translation,comment.
- Include the header row: oare_id,aligned_transliteration,aligned_translation,comment

where each row corresponds to an aligned transliteration and translation
"""
    return prompt

In [16]:
target_id = '004a7dbd-57ce-46f8-9691-409be61c676e' 

train_df = load_data()

prompt = generate_alignment_prompt(target_id,final_df,train_df)

print(prompt)


No empty rows found.
Rows before: 1561, Rows after: 1561

### ALIGNMENT TASK: 004a7dbd-57ce-46f8-9691-409be61c676e

**Full Original Transliteration:**
KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠIB šu-{d}EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUMU a-ta-a 0.3333 ma-na 2 GÍN KÙ.BABBAR SIG₅ i-ṣé-er PUZUR₄-a-šur DUMU a-ta-a a-lá-ḫu-um i-šu iš-tù ḫa-muš-tim ša ì-lí-dan ITU.KAM ša ke-na-tim li-mu-um e-na-sú-in a-na ITU 14 ḫa-am-ša-tim i-ša-qal šu-ma lá iš-qú-ul 1.5 GÍN.TA a-na 1 ma-na-im i-na ITU.1.KAM ṣí-ib-tám ú-ṣa-áb

**Full High-Quality Training Translation:**
Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-Aššur, seal of Puzur-Aššur son of Ataya. Puzur-Aššur son of Ataya owes 22 shekels of good silver to Ali-ahum. Reckoned from the week of Ilī-dan, month of Ša-kēnātim, in the eponymy of Enna-Suen, he will pay in 14 weeks. If he has not paid in time, he will add interest at the rate 1.5 shekel per mina per month.

**Proposed Segments to Align:**
Segme

In [ ]:
SYSTEM_PROMPT = "You are an expert in Old Assyrian cuneiform texts. You follow instructions properly"

In [ ]:
def run_alignment(oare_id,final_df,train_df) -> list[dict]:
    response = client.messages.create(
        model="",
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": generate_alignment_prompt(oare_id,final_df,train_df)}]
    )
    
    if response.stop_reason == "max_tokens":
        print(f"WARNING: Output truncated for {oare_id} — increase max_tokens or split input")
        return [] # don't try to parse incomplete CSV
        
    raw_csv = response.content[0].text.strip()
    if raw_csv.startswith("```"):
        raw_csv = "\n".join(raw_csv.split("\n")[1:])  # strip opening fence
    if raw_csv.endswith("```"):
        raw_csv = raw_csv.rsplit("```", 1)[0]          # strip closing fence
    raw_csv = raw_csv.strip()
    reader = csv.DictReader(io.StringIO(raw_csv))
    
    return list(reader)

def process_all(oare_ids, final_df, train_df, output_path: str = "alignments_master.csv"):
    fieldnames = ["oare_id", "aligned_transliteration", "aligned_translation", "comment"]
    failed = []
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

    for i, oare_id in enumerate(oare_ids):
        print(f"Processing {i+1}/{len(oare_ids)}: {oare_id}")
        try:
            rows = run_alignment(oare_id,final_df,train_df)
            with open(output_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
                writer.writerows(rows)
        except Exception as e:
            failed.append(oare_id)
            print(f"  ERROR on {oare_id}: {e}")
        time.sleep(0.5)  # gentle rate limiting

    print(f"Done. Results in {output_path}")

In [18]:
oare_ids = final_df[final_df['level'] == 'sentence']['id']
print(len(oare_ids.unique()))
print(len(oare_ids))



252
1207
1207


In [15]:
process_all(oare_ids,final_df,train_df)


NameError: name 'SYSTEM_PROMPT' is not defined

In [13]:
def process_all(oare_ids, final_df, train_df, output_path: str = "alignments_master.csv"):
    
    # --- Phase 1: Submit ---
    requests = [
        {
            "custom_id": oare_id,
            "params": {
                "model": "",
                "max_tokens": 4096,
                "system": SYSTEM_PROMPT,
                "messages": [{"role": "user", "content": generate_alignment_prompt(oare_id, final_df, train_df)}]
            }
        }
        for oare_id in oare_ids
    ]
    batch = client.messages.batches.create(requests=requests)
    print(f"Batch submitted: {batch.id}")

    # --- Phase 2: Wait ---
    while True:
        batch = client.messages.batches.retrieve(batch.id)
        print(f"Status: {batch.processing_status} | {batch.request_counts}")
        if batch.processing_status == "ended":
            break
        time.sleep(60)

    # --- Phase 3: Save ---
    fieldnames = ["oare_id", "aligned_transliteration", "aligned_translation", "comment"]
    failed = []

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()

    results = list(client.messages.batches.results(batch.id))  # materialise so we know total
    for i, entry in enumerate(results):
        oare_id = entry.custom_id
        print(f"Saving {i+1}/{len(results)}: {oare_id}")       # ← progress line

        if entry.result.type != "succeeded":
            print(f"  FAILED: {oare_id} — {entry.result.type}")
            failed.append(oare_id)
            continue

        if entry.result.message.stop_reason == "max_tokens":
            print(f"WARNING: truncated for {oare_id}")
            failed.append(oare_id)
            continue

        raw_csv = entry.result.message.content[0].text.strip()
        if raw_csv.startswith("```"):
            raw_csv = "\n".join(raw_csv.split("\n")[1:])
        if raw_csv.endswith("```"):
            raw_csv = raw_csv.rsplit("```", 1)[0]
        raw_csv = raw_csv.strip()

        rows = list(csv.DictReader(io.StringIO(raw_csv)))
        with open(output_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
            writer.writerows(rows)

    if failed:
        print(f"\n{len(failed)} failed: {failed}")
    print(f"Done. Results in {output_path}")

In [6]:
import anthropic
import os
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

batch_id = "msgbatch_01VSrZNuUivjQSZKUwgigopq"#pen("batch_id1.txt").read().strip()

results = list(client.messages.batches.results(batch_id))

# just print first one
entry = results[1]
print(f"id: {entry.custom_id}")
print(f"stop_reason: {entry.result.message.stop_reason}")
print(f"response:\n{entry.result.message.content[0].text}")

id: 009fb838-8038-42bc-ad34-5f795b3840ee
stop_reason: end_turn
response:
I'll analyze the transliteration tokens and align them with the translation segments.

Let me tokenize the transliteration:
0: KIŠIB
1: šu-{d}EN.LÍL
2: DUMU
3: šu-ku-bi-im
4: KIŠIB
5: ṣí-lu-lu
6: DUMU
7: ú-ku
8: i-nu-mì
9: i-dí-a-bu-um
10: a-wa-sú
11: iq-bi-ú
12: 10
13: ma-na
14: KÙ.BABBAR
15: a-na
16: ša-lim-a-šùr
17: i-dí-in
18: um-ma
19: šu-ut-ma
20: i-ṣí-ba-at
21: KÙ-pì-a
22: li-il₅-qé

Segment 0.0: tokens 0-7 (KIŠIB through ú-ku) — the seal introductions. Suggested interval (0,8) looks correct.

Segment 1.0: tokens 8-17 (i-nu-mì through i-dí-in) — the narrative about giving silver. Suggested interval (8,18) — token 17 is i-dí-in, so interval should end at 18. This looks correct.

Segment 2.0: tokens 18-22 (um-ma through li-il₅-qé) — the quoted speech. Suggested interval (18,23) looks correct.

```csv
oare_id,aligned_transliteration,aligned_translation,comment
009fb838-8038-42bc-ad34-5f795b3840ee_0.0,KIŠIB šu-

In [5]:
print(f"total results: {len(results)}")

batch = client.messages.batches.retrieve(batch_id)
print(f"request_counts: {batch.request_counts}")

total results: 252
request_counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=252)


In [7]:
import anthropic
import os
import io
import csv
import pandas as pd

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

batch_id = "msgbatch_01VSrZNuUivjQSZKUwgigopq"
results = list(client.messages.batches.results(batch_id))

fieldnames = ["oare_id", "aligned_transliteration", "aligned_translation", "comment"]
all_rows = []
failed = []

for entry in results:
    oare_id = entry.custom_id

    if entry.result.type != "succeeded":
        failed.append(oare_id)
        continue

    raw = entry.result.message.content[0].text.strip()

    # Extract only the csv block — find the header line and take from there
    lines = raw.split("\n")
    csv_start = next((i for i, l in enumerate(lines) if l.startswith("oare_id,")), None)
    if csv_start is None:
        print(f"No CSV found for {oare_id}")
        failed.append(oare_id)
        continue

    # Strip closing fence if present
    csv_lines = lines[csv_start:]
    csv_lines = [l for l in csv_lines if not l.strip().startswith("```")]
    raw_csv = "\n".join(csv_lines).strip()

    rows = list(csv.DictReader(io.StringIO(raw_csv)))
    all_rows.extend(rows)

df = pd.DataFrame(all_rows, columns=fieldnames)
print(f"Total rows: {len(df)}")
print(f"Failed: {len(failed)}")
df.tail()

No CSV found for f57b8fce-716d-4d0c-a0ba-bdabaa6584ae
Total rows: 1169
Failed: 1


,oare_id,aligned_transliteration,aligned_translation,comment
0,004a7dbd-57ce-46f8-9691-409be61c676e_0,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...,Seal of Mannum-balum-Aššur son of Ṣilli-Adad,seal of Šu-Illil son of Mannum-kī-Aššur
1,004a7dbd-57ce-46f8-9691-409be61c676e_1,0.3333 ma-na 2 GÍN KÙ.BABBAR SIG₅ i-ṣé-er PUZU...,Puzur-Aššur son of Ataya owes 22 shekels of go...,Suggested interval (13-24) adjusted to (12-24)...
2,004a7dbd-57ce-46f8-9691-409be61c676e_2,iš-tù ḫa-muš-tim ša ì-lí-dan ITU.KAM ša ke-na-...,Reckoned from the week of Ilī-dan,month of Ša-kēnātim
3,004a7dbd-57ce-46f8-9691-409be61c676e_3,šu-ma lá iš-qú-ul 1.5 GÍN.TA a-na 1 ma-na-im i...,If he has not paid in time,he will add interest at the rate 1.5 shekel p...
4,009fb838-8038-42bc-ad34-5f795b3840ee_0.0,KIŠIB šu-{d}EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,Seal of Šu-Illil son of Šu-Kūbum,seal of Ṣilūlu son of Uku.


In [8]:
df.tail()

,oare_id,aligned_transliteration,aligned_translation,comment
1164,fded4d2f-81a5-45b3-8d62-74e5ffb2e159_1.0,lu-qú-tám ša ša-lim-a-šur ša i-na ḫa-ḫi-im É i...,As to the goods belonging to Šalim-Aššur that ...,Interval (19-50) is correct; covers the full l...
1165,fe0b8448-f09e-47cf-b1e8-8e8d96531bb3_0.0,a-na a-dí-da a-la-ḫi-im ú ú-ku-ú qí-bi₄-ma um-...,"To Adida, Ali-ahum, and Uku from Puzur-Aššur: ...",First letter segment ending before second addr...
1166,fe0b8448-f09e-47cf-b1e8-8e8d96531bb3_1.0,a-na ú-ku-ú qí-bi₄-ma um-ma PUZUR₄-a-šur-ma la...,To Uku from Puzur-Aššur: Have I not told you a...,Second letter addressed specifically to Uku
1167,0.0,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī-ašrannī son of Sukkalliya has received 3 m...,Interval break at token 13 (il₅-qé) is correct...
1168,1.0,IGI a-šùr-ba-ni DUMU kur-bi-a-a IGI iš-me-DING...,Witnessed by Aššur-bāni son of Kurbiya by Išme...,Witness list tokens 14-21; interval is accurate.


In [9]:
df.to_csv("aligned_dataset.csv")

In [20]:
print(final_df['level'].value_counts())
doc_df = final_df[final_df["level"] == "document"]
doc_df.head()

level
document    1309
sentence    1207
Name: count, dtype: int64


,id,transliteration,translation,level,sentence_idx,original_transliteration,slice_interval,guide_translation,text_at_interval,alignment_status
4,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...,document,NaN,NaN,NaN,NaN,NaN,NO_GUIDE
5,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,<gap> he did not give you a textile. He return...,document,NaN,NaN,NaN,NaN,NaN,NO_GUIDE
9,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...,document,NaN,NaN,NaN,NaN,NaN,NO_GUIDE
10,00f0d841-eb7a-46f8-86fc-bf9fd7d52cbf,um-ma šu-ta-mu-zi e-lá-a en-um-a-šùr ù lá-ma-s...,"From Šu-Tammuzī, Elaya, Ennam-Aššur and Lamass...",document,NaN,NaN,NaN,NaN,NaN,NO_GUIDE
11,0123a9b9-e81e-4d7a-a79b-10e7c0aacbb9,KIŠIB a-lá-ḫi-im KIŠIB a-li-li KIŠIB a-bi₄-lá ...,"Seal of Ali-ahum, seal of Ali-ilī, seal of Abi...",document,NaN,NaN,NaN,NaN,NaN,NO_GUIDE


In [22]:
df = pd.read_csv("aligned_dataset.csv")
df.head()

,Unnamed: 0,oare_id,aligned_transliteration,aligned_translation,comment
0,0,004a7dbd-57ce-46f8-9691-409be61c676e_0,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...,Seal of Mannum-balum-Aššur son of Ṣilli-Adad,seal of Šu-Illil son of Mannum-kī-Aššur
1,1,004a7dbd-57ce-46f8-9691-409be61c676e_1,0.3333 ma-na 2 GÍN KÙ.BABBAR SIG₅ i-ṣé-er PUZU...,Puzur-Aššur son of Ataya owes 22 shekels of go...,Suggested interval (13-24) adjusted to (12-24)...
2,2,004a7dbd-57ce-46f8-9691-409be61c676e_2,iš-tù ḫa-muš-tim ša ì-lí-dan ITU.KAM ša ke-na-...,Reckoned from the week of Ilī-dan,month of Ša-kēnātim
3,3,004a7dbd-57ce-46f8-9691-409be61c676e_3,šu-ma lá iš-qú-ul 1.5 GÍN.TA a-na 1 ma-na-im i...,If he has not paid in time,he will add interest at the rate 1.5 shekel p...
4,4,009fb838-8038-42bc-ad34-5f795b3840ee_0.0,KIŠIB šu-{d}EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,Seal of Šu-Illil son of Šu-Kūbum,seal of Ṣilūlu son of Uku.


In [27]:
df = df.rename(columns={
    "aligned_transliteration": "transliteration",
    "aligned_translation": "translation"
})

combined = pd.concat([df, df], ignore_index=True)

In [28]:
len(combined)

combined.to_csv("perfected_aligned_train.csv")